In [1]:
import os
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)

print("TensorFlow Version :", tf.__version__)

TensorFlow Version : 2.16.1


In [2]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

dataset_path = "../../datasets/food-101/images"

print(dataset_path)
print(os.path.exists(dataset_path))

../../datasets/food-101/images
True


In [3]:
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print("Datasets Loaded")

Found 101000 files belonging to 101 classes.
Using 80800 files for training.
Found 101000 files belonging to 101 classes.
Using 20200 files for validation.
Datasets Loaded


In [4]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)

print("Dataset Ready")

Dataset Ready


In [5]:
model = load_model("../saved_models/efficientnetb0_finetuned.keras")

print("Fine-tuned model loaded successfully")

Fine-tuned model loaded successfully


In [13]:
base_model = model.layers[2]

# First make ALL layers trainable
base_model.trainable = True

for layer in base_model.layers:
    layer.trainable = True

print("All Trainable:",
      sum(layer.trainable for layer in base_model.layers))

All Trainable: 238


In [14]:
for layer in base_model.layers[:-60]:
    layer.trainable = False

print("Trainable Layers:",
      sum(layer.trainable for layer in base_model.layers))

Trainable Layers: 60


In [15]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=5e-6
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Advanced Fine-Tune Model Compiled")

Advanced Fine-Tune Model Compiled


In [16]:
checkpoint_phase3 = ModelCheckpoint(
    "../saved_models/efficientnetb0_advanced.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

In [17]:
early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

csv_logger = CSVLogger(
    "../saved_models/training_log_advanced.csv"
)

print("Callbacks Ready")

Callbacks Ready


In [18]:
ADVANCED_EPOCHS = 20

history_advanced = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=ADVANCED_EPOCHS,
    callbacks=[
        checkpoint_phase3,
        early_stop,
        reduce_lr,
        csv_logger
    ],
    verbose=1
)

Epoch 1/20
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6444 - loss: 1.3502
Epoch 1: val_accuracy improved from None to 0.72356, saving model to ../saved_models/efficientnetb0_advanced.keras

Epoch 1: finished saving model to ../saved_models/efficientnetb0_advanced.keras
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 5646s 1s/step - accuracy: 0.6490 - loss: 1.3218 - val_accuracy: 0.7236 - val_loss: 1.0476 - learning_rate: 5.0000e-06
Epoch 2/20
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 0s 569ms/step - accuracy: 0.6563 - loss: 1.2909
Epoch 2: val_accuracy improved from 0.72356 to 0.72723, saving model to ../saved_models/efficientnetb0_advanced.keras

Epoch 2: finished saving model to ../saved_models/efficientnetb0_advanced.keras
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 3342s 662ms/step - accuracy: 0.6600 - loss: 1.2761 - val_accuracy: 0.7272 - val_loss: 1.0311 - learning_rate: 5.0000e-06
Epoch 3/20
5050/5050 ━━━━━━━━━━━━━━━━━━━━ 0s 623ms/step - accuracy: 0.6664 - loss: 1.2580
Epoch 3: val_accuracy improved from 0